<a href="https://colab.research.google.com/github/Ahbar1999/mtp-pimsimulator/blob/main/pimsimulator_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Code Summary**


*   In the (wear count and address translation) arrays and indices of the arrays are logical array ids   
*   For intra crossbar wear levelling, row swapping is being implemented as given in the TIME paper, DRCTL implements intra crossbar wear levelling by doing continuous shifting of row, columns which does not take into account the dynamic ageing
*   For inter crossbar wear levelling, TIWL's approach is being implemented
*   Currently the writes are being accumulated per xbar, we need to accumulate writes per cell

Implmentation References:
*   ODLPIM: https://past.date-conference.com/proceedings-archive/2023/DATA/357.pdf
*   DRCTL:  https://ieeexplore.ieee.org/document/10764631
*   TIME:   https://dl.acm.org/doi/10.1145/3195970.3196071

**TO-DO**

*   Find the statistic used in the paper for determining performance of intra crossbar wear levelling, use it find results
*   Currently only the gradient weights i.e. memory thats being written to, is monitored, we dont know how the other memory access i.e reads are taking place, also when you change address mappings, how are the memory accesses dependent on changed mappings get affected ?



**Learnings/Experiments**

*   Row swapping intervals/ min-max ratio thresholds to swap:
      Using smaller min-max ratio and smaller swapping intervals did not result in smaller max-count for rows because of ???
*        



PROBLEMS:
  1- There's a discrepency between row write counters and crossbar write counters ; fix it

  Answer:
      Why the "Contradiction" happened (Rows vs. Xbars)
      Row Plot: Visualizes row_write_counts. This is an accumulative array that never resets. It captured the writes at Physical 50-99 and the writes at Physical 0-49. Because the Intra-block remapping (row_mappings) is working, it spread those writes across all rows within those blocks. To your eye, it looked "fully colored" because the rows were active in the active blocks.
      
      Crossbar Plot: Visualizes twc_counters. This showed the total wear per block. Because of the Double Registration, you had massive unused space (Physical 100-511). The imshow function likely scaled the colors such that the active blocks looked like "points" in a sea of black zeros.

In [ ]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from collections import defaultdict

In [ ]:
class PIMSimulator:
    def __init__(self,
                 num_crossbars=1024,
                 crossbar_size=(128, 128),
                 bits_per_cell=1,
                 weight_bits=8,
                 wear_levelling_enabled=True,
                 intra_xbar_wear_levelling_enabled=True,
                 slc_bits=1,
                 mlc_bits=4,
                 slc_mlc_ratio=0.1):
        """Config params"""
        self.num_crossbars = num_crossbars
        self.crossbar_rows, self.crossbar_cols = crossbar_size
        self.bits_per_cell = bits_per_cell  # bit resolution of each reram cell(regardless of whether cell is slc or mlc)
        # (quantization)bits per weight
        self.weight_bits = weight_bits
        self.wear_levelling_enabled = wear_levelling_enabled
        self.intra_xbar_wear_levelling_enabled = intra_xbar_wear_levelling_enabled

        # Calculate how many weights fit in one crossbar
        self.cells_per_crossbar = self.crossbar_rows * self.crossbar_cols
        self.weights_per_crossbar = self.cells_per_crossbar // self.weight_bits

        # interval write counters for each crossbar
        self.iwc_counters = np.zeros(num_crossbars, dtype=np.uint32)
        # total write counters for each crossbar
        self.twc_counters = np.zeros(num_crossbars, dtype=np.uint64)

        # addressing granularity is a crossbar
        # LAID -> PAID: initial mapping is i -> i
        self.access_count = 0                   # xbar memory accesses
        self.access_count_for_rows = 0
        self.remapping_interval = 1000

        self.tensor_registry = {}           # tensor_name -> segment_list
        # the following map should be changed to xbar_base_addr -> xbar_object
        # the crossbar object will have an address field
        self.laid_to_paid_map = {}          # logical to physical xbar ids
        self.next_laid = 0                  # next free LAID(that maps to physical id)
        self.next_physical_crossbar = 0     # next free PAID(of the actual crossbar)

        # row mapping code
        # write count 2D array for xbar * rows_per_xbar
        # [physical xbar id][physical row id] => logical row id: Wrong mapping
        # [physical xbar id][logical row id] => physical row id

        self.row_write_counts = np.zeros((num_crossbars, self.crossbar_rows), dtype=np.int64)
        self.row_mappings = [list(range(self.crossbar_rows)) for _ in range(num_crossbars)]

        # row max / min
        self.row_swap_threshold = 2.0
        # swap rows every row_swap_interval writes
        self.row_swap_interval = 10000
        self.total_row_swaps =0

        # param update frequency counters
        self.param_update_counter: dict['str', dict['str', int]] = {}
        # gradient magnitudes
        self.grad_updates = {}

        # [NEW] slc/mlc config
        self.slc_bits = slc_bits
        self.mlc_bits = mlc_bits
        self.num_slc_xbars = int(num_crossbars * slc_mlc_ratio)
        self.num_mlc_xbars = num_crossbars - self.num_slc_xbars

        # Physical Pools
        self.slc_pool = list(range(0, self.num_slc_xbars))
        self.mlc_pool = list(range(self.num_slc_xbars, num_crossbars))

        # method to use to log memory accesses
        self.logger = None

        print(f"Crossbar config: {crossbar_size}, {self.weights_per_crossbar} weights per crossbar")
        if not self.wear_levelling_enabled:
          print("Wear levelling disabled")

    def register_model(self, model):
        """Build collision-free mapping for all tensors once before running the model"""
        for name, param in model.named_parameters():
            # generate mapping for trainable parameters
            if param.requires_grad:
                # segment are basically individual xbar mappings for the whole tensor
                segments = self._partition_tensor(name, param)
                self.tensor_registry[name] = segments

    def _partition_tensor(self, tensor_name, tensor):
        """Distribute tensor weights to xbars"""
        tensor_size = tensor.numel()
        crossbars_needed = math.ceil(tensor_size / self.weights_per_crossbar)
        # print(f"Number of Crossbars needed to deploy {tensor_name}: {crossbars_needed}")

        segments = []
        remaining_elements = tensor_size

        for i in range(crossbars_needed):
            laid = self.next_laid
            if self.next_physical_crossbar >= self.num_crossbars:
                raise ValueError(f"Not enough crossbars for tensor {tensor_name}, exiting program.")

            paid = self.next_physical_crossbar
            elements = min(remaining_elements, self.weights_per_crossbar)

            '''
              removed the following field from segments object
              'logical_offset': tensor_size - remaining_elements # not required; what even is it ?
            '''
            segments.append({
                'laid': laid,
                'paid': paid,
                'elements': elements,
            })

            # Initial i -> i mapping
            self.laid_to_paid_map[laid] = paid

            self.next_laid += 1
            self.next_physical_crossbar += 1
            remaining_elements -= elements

        return segments

    def log_memory_access(self, tensor_name, k_swaps=-1, is_write=True):
        """Accurate logging considering tensor size and crossbar capacity"""
        if tensor_name not in self.tensor_registry:
            return

        if tensor_name not in self.param_update_counter:
            self.param_update_counter[tensor_name] = {}

        if not is_write:
          if 'read' not in self.param_update_counter[tensor_name]:
            self.param_update_counter[tensor_name]['read'] = 0
          self.param_update_counter[tensor_name]['read'] += 1
        else:
          if 'write' not in self.param_update_counter[tensor_name]:
            self.param_update_counter[tensor_name]['write'] = 0
          self.param_update_counter[tensor_name]['write'] += 1

        xbars = self.tensor_registry[tensor_name]

        for xbar in xbars:
            laid = xbar['laid']
            current_paid = self.laid_to_paid_map[laid]  # address translation
            elements = xbar['elements'] # an element is a single param

            writes_per_element = self.weight_bits // self.bits_per_cell
            total_writes = elements * writes_per_element

            # record interval write count for current xbar
            # self.iwc_counters[current_paid] += total_writes
            self.iwc_counters[laid] += total_writes

            self._log_row_level_writes(laid, elements, total_writes)

        self.access_count += 1
        # this could be problematic
        self.access_count_for_rows += 1

        if self.access_count % self.remapping_interval == 0:
            self.perform_remapping(k_swaps)

        if self.access_count_for_rows % self.remapping_interval == 0:
            self.perform_row_remapping()

    def _log_row_level_writes(self, crossbar_laid, elements, total_writes):
        weights_per_row = self.crossbar_cols * (self.bits_per_cell // self.weight_bits)
        if weights_per_row <= 0:
            weights_per_row = self.crossbar_cols // 8  # Fallback: assume 8-bit weights

        rows_affected = math.ceil(elements / weights_per_row)
        writes_per_row = total_writes // rows_affected if rows_affected > 0 else total_writes

        # assert(rows_affected <= 128)

        for logical_row in range(min(rows_affected, self.crossbar_rows)):
            physical_xbar = self.laid_to_paid_map[crossbar_laid]
            physical_row = self.row_mappings[physical_xbar][logical_row]

            # assert(physical_xbar < 128 and physical_row < 128)
            self.row_write_counts[physical_xbar][physical_row] += writes_per_row

    # this has to be performed within each xbar
    def perform_row_remapping(self):
        if not self.wear_levelling_enabled or not self.intra_xbar_wear_levelling_enabled:
          return

        """Perform new swapping within crossbars to balance row-level wear"""
        """Swaps 2 highest and lowest utilized rows in xbars"""
        # print("Performing Row Remapping...")

        # crossbar_id <-> laid
        for crossbar_id in range(self.num_crossbars):
          row_counts = self.row_write_counts[crossbar_id]

          if np.sum(row_counts) == 0:
            continue

          max_wear_physical = np.max(row_counts)  # maximum write count
          min_wear_physical = np.min(row_counts)  # minimum write count

          if min_wear_physical == 0 or (max_wear_physical / min_wear_physical) < self.row_swap_threshold:
            continue

          # get logical id of the rows with min, max write counts
          max_logical = np.argmax(row_counts)
          min_logical = np.argmin(row_counts)

          # # swap physical ids of the min/max rows
          # self.row_mappings[crossbar_id][max_logical] = self.row_mappings[crossbar_id][min_logical]
          # self.row_mappings[crossbar_id][min_logical] = self.row_mappings[crossbar_id][max_logical]

          phys_max = self.row_mappings[crossbar_id][max_logical]
          phys_min = self.row_mappings[crossbar_id][min_logical]

          # Swap min/max physical ids
          self.row_mappings[crossbar_id][max_logical] = phys_min
          self.row_mappings[crossbar_id][min_logical] = phys_max


          self.total_row_swaps += 1
          # print("row swapped")
          # # exit(0)

          # Optional: Log the swap
          if self.access_count % (self.row_swap_interval * 10) == 0:  # Log occasionally
            print(f"Row swap in crossbar {crossbar_id}: "
                  f"logical {max_logical}→physical {min_wear_physical}, "
                  f"logical {min_logical}→physical {max_wear_physical} "
                  f"(wear ratio: {max_wear_physical/min_wear_physical:.2f})")

    def perform_remapping(self, k=-1):
        # print("Updating write counters")
        # Update Total Wear Count (TWC)
        for laid, iwc in enumerate(self.iwc_counters):
            paid = self.laid_to_paid_map.get(laid, laid)
            self.twc_counters[paid] += iwc
            # self.iwc_counters[laid] = 0  # Reset interval counter -> this shouldnt be here

        if not self.wear_levelling_enabled:
            # print("wear levelling has been disabled!")
            self.iwc_counters.fill(0)
            return

        # print("Performing Remapping...")
        # Sort crossbars by wear level (TWC)
        wear_sorted_crossbars = sorted(enumerate(self.twc_counters), key=lambda x: x[1])

        # Sort logical crossbars by hotness (recent writes)
        hotness_sorted = sorted(enumerate(self.iwc_counters), key=lambda x: x[1], reverse=True)

        # Rebuild mapping: hottest logical → coolest physical
        new_mapping = self.laid_to_paid_map.copy()
        # (hot laid, hotness value)
        cap = int(len(wear_sorted_crossbars) * (k / 100.0))
        # if k != -1:
        #   print(f"remapping top {k}% xbars")
        # else:
        #   print("remapping all xbars")

        for i, (hot_laid, _) in enumerate(hotness_sorted):
            if i != -1 and i >= cap:
              # restrict remapping to only top k%
              break

            cool_paid, _ = wear_sorted_crossbars[i]  # Least worn physical crossbar
            new_mapping[hot_laid] = cool_paid

        # print("change in mappings: ")
        # for laid in new_mapping:
        #   if self.laid_to_paid_map[laid] != new_mapping[laid]:
        #     print(f"{laid}: {self.laid_to_paid_map[laid]} -> {new_mapping[laid]}")

        self.laid_to_paid_map = new_mapping
        # print(f"New mapping: {self.laid_to_paid_map}")
        # reset counters
        self.iwc_counters.fill(0)

    def get_crossbar_utilization_stats(self):
        """Analyze crossbar utilization patterns"""
        stats = {
            'total_crossbars': self.num_crossbars,
            'weights_per_crossbar': self.weights_per_crossbar,
            'active_crossbars': np.count_nonzero(self.iwc_counters),
            'avg_wear': np.mean(self.twc_counters),
            'max_wear': np.max(self.twc_counters),
            'wear_imbalance': np.max(self.twc_counters) / (np.min(self.twc_counters) + 1)
        }
        return stats

    def get_row_wear_statistics(self):
      """
      Row-level wear statistics after considering intra-crossbar row swapping.
      Also returns crossbar-level stats derived from row data and reduction metrics.
      Relies on:
        - self.row_write_counts: shape (num_crossbars, crossbar_rows), physical rows
        - self.row_mappings: logical->physical mapping (not needed to read counts since counts are stored on physical rows)
        - self._total_row_swaps: optional counter (defaults to 0 if absent)
      """
      wear_rows = self.row_write_counts  # shape (C, R), physical rows
      row_sums_per_xbar = wear_rows.sum(axis=1)
      active_xbar_mask = row_sums_per_xbar > 0
      active_rows = wear_rows[active_xbar_mask]

      if active_rows.size == 0:
          return {
              'max_row_wear': 0,
              'min_row_wear': 0,
              'avg_row_wear': 0.0,
              'row_wear_imbalance': 1.0,
              'active_crossbars': 0,
              'total_row_swaps': int(getattr(self, 'total_row_swaps', 0)),
              # Aggregates per crossbar from row totals
              'max_xbar_wear_from_rows': 0,
              'min_xbar_wear_from_rows': 0,
              'avg_xbar_wear_from_rows': 0.0,
              'xbar_wear_imbalance_from_rows': 1.0
          }

      flat = active_rows.flatten()
      nonzero = flat[flat > 0]
      max_row = int(flat.max())
      min_row_active = int(nonzero.min()) if nonzero.size > 0 else 0
      avg_row = float(nonzero.mean()) if nonzero.size > 0 else 0.0
      row_imbalance = (max_row / min_row_active) if min_row_active > 0 else 1.0

      xbar_totals = active_rows.sum(axis=1)
      max_xbar = int(xbar_totals.max())
      min_xbar_active = int(xbar_totals[xbar_totals > 0].min()) if np.any(xbar_totals > 0) else 0
      avg_xbar = float(xbar_totals.mean()) if xbar_totals.size > 0 else 0.0
      xbar_imbalance = (max_xbar / min_xbar_active) if min_xbar_active > 0 else 1.0


      return {
          'max_row_wear': max_row,
          'min_row_wear': min_row_active,
          'avg_row_wear': avg_row,
          'row_wear_imbalance': row_imbalance,
          'active_crossbars': int(active_xbar_mask.sum()),
          'total_row_swaps': int(getattr(self, 'total_row_swaps', 0)),
          'max_xbar_wear_from_rows': max_xbar,
          'min_xbar_wear_from_rows': min_xbar_active,
          'avg_xbar_wear_from_rows': avg_xbar,
          'xbar_wear_imbalance_from_rows': xbar_imbalance
      }

    def _get_capacity(self, bits_per_cell):
      cells_per_xbar = self.crossbar_rows * self.crossbar_cols
      # How many weights fit depends on cell precision
      return (cells_per_xbar * bits_per_cell) // self.weight_bits

    def redistribute_tensors_by_magnitude(self, model, k_percent=10):
      """
      Runs once after initial epochs.
      Wipes existing registry and re-maps based on current weight magnitudes.
      """
      self.tensor_registry = {}
      self.laid_to_paid_map = {}
      self.next_laid = 0
      current_slc_idx = 0
      current_mlc_idx = 0

      print(f"Redistributing model: Top {k_percent}% to SLC ({self.slc_bits}-bit), rest to MLC ({self.mlc_bits}-bit)")

      for name, param in model.named_parameters():
          if not param.requires_grad:
            continue

          w = param.data.abs().view(-1)
          num_elements = w.size(0)

          # 1. Determine Threshold
          k_elements = int(num_elements * (k_percent / 100.0))
          if k_elements > 0:
              threshold = torch.kthvalue(w, num_elements - k_elements + 1).values.item()
          else:
              threshold = float('inf')

          # 2. Partition into SLC and MLC counts
          slc_elements = (w >= threshold).sum().item()
          mlc_elements = num_elements - slc_elements

          segments = []

          # 3. Map SLC Segments
          segments.extend(self._allocate_segments(slc_elements, mode='SLC', start_pool_idx=current_slc_idx))
          current_slc_idx += math.ceil(slc_elements / self._get_capacity(self.slc_bits))

          # 4. Map MLC Segments
          segments.extend(self._allocate_segments(mlc_elements, mode='MLC', start_pool_idx=current_mlc_idx))
          current_mlc_idx += math.ceil(mlc_elements / self._get_capacity(self.mlc_bits))

          self.tensor_registry[name] = segments

      print(f"Mapping complete. SLC Xbars used: {current_slc_idx}/{self.num_slc_xbars}")

    def _allocate_segments(self, count, mode, start_pool_idx):
        segments = []
        bits = self.slc_bits if mode == 'SLC' else self.mlc_bits
        pool = self.slc_pool if mode == 'SLC' else self.mlc_pool
        capacity = self._get_capacity(bits)

        remaining = count
        local_idx = start_pool_idx

        while remaining > 0:
            if local_idx >= len(pool):
                raise ValueError(f"Out of {mode} crossbars!")

            elements_in_this_xbar = min(remaining, capacity)
            laid = self.next_laid
            paid = pool[local_idx]

            segments.append({
                'laid': laid,
                'paid': paid,
                'elements': elements_in_this_xbar,
                'bits_per_cell': bits,
                'mode': mode
            })

            self.laid_to_paid_map[laid] = paid
            self.next_laid += 1
            local_idx += 1
            remaining -= elements_in_this_xbar

        return segments

    def log_memory_access_v2(self, tensor_name, k_swaps=-1, is_write=True):
      if tensor_name not in self.tensor_registry:
        return

      for segment in self.tensor_registry[tensor_name]:
          laid = segment['laid']
          paid = self.laid_to_paid_map[laid]

          # Weight bits / bits_per_cell = pulses needed per weight
          pulses_per_element = self.weight_bits / segment['bits_per_cell']
          total_writes = int(segment['elements'] * pulses_per_element)

          self.iwc_counters[laid] += total_writes
          self.twc_counters[paid] += total_writes # Simplification for total wear

          # Log row-level wear
          weights_per_row = (self.crossbar_cols * segment['bits_per_cell']) // self.weight_bits
          rows_affected = math.ceil(segment['elements'] / max(1, weights_per_row))
          writes_per_row = total_writes // max(1, rows_affected)

          self.access_count += 1

          for row_idx in range(min(rows_affected, self.crossbar_rows)):
              self.row_write_counts[paid][row_idx] += writes_per_row

      self.access_count += 1
      if self.access_count % self.remapping_interval == 0:
          self.perform_remapping(k_swaps)

      if self.access_count_for_rows % self.remapping_interval == 0:
          self.perform_row_remapping()



In [ ]:
class PIMTrainingHook:
    def __init__(self, pim_simulator, k_swaps=-1):
        self.pim_sim = pim_simulator
        self.hooks = []
        self.model_params = {} # To store model parameters by name
        self.k_swaps= k_swaps

    def forward_hook(self, module, input, output, name):
        """Log forward pass memory accesses"""
        tensor_name = f"{name}.weight"

        # self.pim_sim.log_memory_access(tensor_name, k_swaps=self.k_swaps, is_write=False)
        if not self.pim_sim.logger:
          raise ValueError("logger not set")

        self.pim_sim.logger(tensor_name, k_swaps=self.k_swaps, is_write=True)

    def backward_hook(self, grad, name):
      tensor_name = f"{name}.weight"

      # log param update magnitude
      if grad is not None:
        norm = torch.norm(grad).item()
        if name not in self.pim_sim.grad_updates:
          self.pim_sim.grad_updates[name] = []
        self.pim_sim.grad_updates[name].append(norm)

      if not self.pim_sim.logger:
          raise ValueError("logger not set")
      # self.pim_sim.log_memory_access(tensor_name, k_swaps=self.k_swaps, is_write=True)
      self.pim_sim.logger(tensor_name, k_swaps=self.k_swaps, is_write=True)

    def register_hooks(self, model):
        self.pim_sim.register_model(model)

        # hook (only)trainable params for recording write access
        for name, module in model.named_modules():
            if hasattr(module, 'weight') and module.weight.requires_grad:
                # self.model_params[name + '.weight'] = module.weight # Store parameter
                # hook = lambda grad, name=name: self.pim_sim.log_memory_access(
                #     f"{name}.weight", k_swaps=self.k_swaps, is_write=True
                # )
                # register read hoook
                module.register_forward_hook(lambda m, i, o, n=name: self.forward_hook(m, i, o, n))

                # register write hook(backprop)
                module.weight.register_hook(lambda g, n=name: self.backward_hook(g, n))

    '''
    def register_hooks(self, model):
        self.pim_sim.register_model(model)

        # hook (only)trainable params for recording write access
        for name, module in model.named_modules():
            if hasattr(module, 'weight') and module.weight.requires_grad:
                # self.model_params[name + '.weight'] = module.weight # Store parameter
                hook = lambda grad, name=name: self.pim_sim.log_memory_access(
                    f"{name}.weight", k_swaps=self.k_swaps, is_write=True
                )

                # we are only registering hooks for backprop
                module.weight.register_hook(hook)
    def forward_hook(self, module, input, output, name):
        """Log forward pass memory accesses"""
        if hasattr(module, 'weight'):
            self.pim_sim.log_memory_access(
                module.weight, is_write=False, operation=f"forward_{name}"
            )
        if isinstance(output, torch.Tensor):
            self.pim_sim.log_memory_access(
                output, is_write=True, operation=f"output_{name}"
            )

    def parameter_hook(self, grad, name):
        """Log parameter update with accurate crossbar mapping"""
        if grad is not None:
            param = self.get_parameter_by_name(name)  # Get corresponding parameter tensor

            # Log both gradient computation and parameter update
            self.pim_sim.log_memory_access(grad, is_write=False, operation=f"grad_read_{name}")
            self.pim_sim.log_memory_access(param, is_write=True, operation=f"param_write_{name}")

            # Print mapping info for debugging
            mappings = self.pim_sim.map_tensor_to_crossbars(param)
            total_crossbars = len(mappings)
            total_writes = sum(m['elements'] for m in mappings) * self.pim_sim.weight_bits

            # print(f"Parameter {name}: {param.shape} -> {total_crossbars} crossbars, {total_writes} total writes")
    '''
    def get_parameter_by_name(self, name):
        """Retrieve parameter tensor by name"""
        # The name in the hook is just the "module name + '.weight'"
        param_name = name + '.weight'
        return self.model_params.get(param_name)

    def cleanup(self):
        """Remove all hooks"""
        for hook in self.hooks:
            hook.remove()

In [ ]:
import torch

class PIMFreezeManager:
    def __init__(self, pim_sim):
        self.pim_sim = pim_sim
        self.param_masks = {}

    def generate_freeze_masks(self, model):
        """
        Call this immediately after redistribute_tensors_by_magnitude.
        Creates a mask where SLC elements = 1 and MLC elements = 0.
        """
        for name, param in model.named_parameters():
            if name in self.pim_sim.tensor_registry:
                # Initialize a mask of zeros (assume MLC/frozen by default)
                mask = torch.zeros_like(param, dtype=torch.float32)

                # Flatten to map indices easily
                flat_mask = mask.view(-1)
                offset = 0

                # Iterate through the segments stored in your registry
                for segment in self.pim_sim.tensor_registry[name]:
                    num_el = segment['elements']
                    # If the segment is SLC, set those indices to 1 (active)
                    if segment.get('mode') == 'SLC':
                        flat_mask[offset : offset + num_el] = 1.0
                    offset += num_el

                self.param_masks[name] = mask

    def apply_freeze(self, model):
        """
        Call this after loss.backward() but before optimizer.step().
        It physically prevents the 'frozen' weights from updating.
        """
        for name, param in model.named_parameters():
            if name in self.param_masks and param.grad is not None:
                # Multiply gradient by mask (MLC gradients become 0)
                param.grad.mul_(self.param_masks[name])

In [ ]:
def get_row_wear_statistics(self):
    """
    Row-level wear statistics after considering intra-crossbar row swapping.
    Also returns crossbar-level stats derived from row data and reduction metrics.
    Relies on:
      - self.row_write_counts: shape (num_crossbars, crossbar_rows), physical rows
      - self.row_mappings: logical->physical mapping (not needed to read counts since counts are stored on physical rows)
      - self._total_row_swaps: optional counter (defaults to 0 if absent)
    """
    wear_rows = self.row_write_counts  # shape (C, R), physical rows
    row_sums_per_xbar = wear_rows.sum(axis=1) # for each xbar sum the write count of all rows
    active_xbar_mask = row_sums_per_xbar > 0
    # get only the rows that have atleast one write count
    active_rows = wear_rows[active_xbar_mask]

    if active_rows.size == 0:
        return {
            'max_row_wear': 0,
            'min_row_wear': 0,
            'avg_row_wear': 0.0,
            'row_wear_imbalance': 1.0,
            'active_crossbars': 0,
            'total_row_swaps': self.total_row_swaps,
            # Aggregates per crossbar from row totals
            'max_xbar_wear_from_rows': 0,
            'min_xbar_wear_from_rows': 0,
            'avg_xbar_wear_from_rows': 0.0,
            'xbar_wear_imbalance_from_rows': 1.0
        }

    flat = active_rows.flatten()
    nonzero = flat[flat > 0]  # rows with non zero write count
    max_row = int(flat.max())
    min_row_active = int(nonzero.min()) if nonzero.size > 0 else 0
    avg_row = float(nonzero.mean()) if nonzero.size > 0 else 0.0
    row_imbalance = (max_row / min_row_active) if min_row_active > 0 else 1.0

    # per xbar stats
    xbar_totals = active_rows.sum(axis=1)
    max_xbar = int(xbar_totals.max())
    min_xbar_active = int(xbar_totals[xbar_totals > 0].min()) if np.any(xbar_totals > 0) else 0
    avg_xbar = float(xbar_totals.mean()) if xbar_totals.size > 0 else 0.0
    xbar_imbalance = (max_xbar / min_xbar_active) if min_xbar_active > 0 else 1.0


    return {
        'max_row_wear': max_row,
        'min_row_wear': min_row_active,
        'avg_row_wear': avg_row,
        'row_wear_imbalance': row_imbalance,
        'active_crossbars': int(active_xbar_mask.sum()),
        'total_row_swaps': self.total_row_swaps,
        'max_xbar_wear_from_rows': max_xbar,
        'min_xbar_wear_from_rows': min_xbar_active,
        'avg_xbar_wear_from_rows': avg_xbar,
        'xbar_wear_imbalance_from_rows': xbar_imbalance
    }


def plot_row_wear_heatmap(pim_simulator_no_tiwl, pim_simulator_with_tiwl,
                          grid_shape=(16, 32), rows_to_show=128, save_path=None):
    """
    Plot side-by-side heatmaps of per-row wear (physical rows) for two simulators.
    Each heatmap stacks rows_to_show rows of each crossbar into a grid:
      final shape ~ (grid_rows*rows_to_show, grid_cols).
    Also prints row-level statistics and percentage reductions.

    Args:
        pim_simulator_no_tiwl: simulator without wear levelling enabled
        pim_simulator_with_tiwl: simulator with wear levelling enabled
        grid_shape: (grid_rows, grid_cols) layout of crossbars
        rows_to_show: number of rows to display per crossbar block
        save_path: optional file path to save the figure
    """
    import numpy as np
    import matplotlib.pyplot as plt

    def build_grid(sim):
        wear_rows = sim.row_write_counts  # shape (C, R), physical rows
        C, R = wear_rows.shape
        gr, gc = grid_shape
        total = gr * gc
        take = min(C, total)
        # Take first 'take' crossbars and first 'rows_to_show' rows
        block = wear_rows[:take, :min(rows_to_show, R)]  # (take, rows_to_show)
        # Pad vertically if rows_to_show > R for consistent panel size
        if block.shape[1] < rows_to_show:
            pad = np.zeros((block.shape[0], rows_to_show - block.shape[1]), dtype=block.dtype)
            block = np.concatenate([block, pad], axis=1)

        blocks = []
        idx = 0
        for _ in range(gr):
            row_blocks = []
            for _ in range(gc):
                if idx < take:
                    # shape (rows_to_show, 1): column is the crossbar index in this grid col
                    row_blocks.append(block[idx][:, None])
                else:
                    row_blocks.append(np.zeros((rows_to_show, 1), dtype=block.dtype))
                idx += 1
            row_strip = np.concatenate(row_blocks, axis=1)  # (rows_to_show, gc)
            blocks.append(row_strip)
        return np.concatenate(blocks, axis=0)  # (gr*rows_to_show, gc)

    grid_no = build_grid(pim_simulator_no_tiwl)
    grid_yes = build_grid(pim_simulator_with_tiwl)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    fig.patch.set_alpha(1.0)
    # with plt.style.context('default'):
    im1 = ax1.imshow(grid_no, cmap='hot',  interpolation='nearest', aspect='auto')
    ax1.set_title('Row Wear Without TIWL')
    ax1.set_xlabel('Crossbar column index')
    ax1.set_ylabel('Row index within each crossbar block')
    cbar1 = plt.colorbar(im1, ax=ax1)
    cbar1.set_label('Writes per row', rotation=270, labelpad=20)

    im2 = ax2.imshow(grid_yes, cmap='hot', interpolation='nearest', aspect='auto')
    ax2.set_title('Row Wear With TIWL')
    ax2.set_xlabel('Crossbar column index')
    ax2.set_ylabel('Row index within each crossbar block')
    cbar2 = plt.colorbar(im2, ax=ax2)
    cbar2.set_label('Writes per row', rotation=270, labelpad=20)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    # Row-level statistics and reductions
    stats_no = pim_simulator_no_tiwl.get_row_wear_statistics()
    stats_yes = pim_simulator_with_tiwl.get_row_wear_statistics()

    def pct_reduction(a, b):
        return 0.0 if a <= 0 else 100.0 * (a - b) / a

    max_row_reduction = pct_reduction(stats_no.get('max_row_wear', 0), stats_yes.get('max_row_wear', 0))
    imbalance_reduction = pct_reduction(stats_no.get('row_wear_imbalance', 1.0), stats_yes.get('row_wear_imbalance', 1.0))
    max_xbar_from_rows_reduction = pct_reduction(
        stats_no.get('max_xbar_wear_from_rows', 0), stats_yes.get('max_xbar_wear_from_rows', 0)
    )
    min_xbar_from_rows_reduction = pct_reduction(
        stats_no.get('min_xbar_wear_from_rows', 0), stats_yes.get('min_xbar_wear_from_rows', 0)
    )
    avg_xbar_from_rows_reduction = pct_reduction(
        stats_no.get('avg_xbar_wear_from_rows', 0), stats_yes.get('avg_xbar_wear_from_rows', 0)
    )

    print("Row-level wear statistics (derived from per-row counts):")
    print(f"Without TIWL - max_row: {stats_no.get('max_row_wear', 0):,}, min_row: {stats_no.get('min_row_wear', 0):,},"
          f"avg_row: {stats_no.get('avg_row_wear', 0.0):.2f}, "
          # f"imbalance (max/min+1): {stats_no.get('row_wear_imbalance', 1.0):.2f}"
          )

    print(f"With TIWL    - max_row: {stats_yes.get('max_row_wear', 0):,}, min_row: {stats_yes.get('min_row_wear', 0):,},"
          f"avg_row: {stats_yes.get('avg_row_wear', 0.0):.2f}, "
          # f"imbalance (max/min+1): {stats_yes.get('row_wear_imbalance', 1.0):.2f}"
          )

    # print(f"Reduction - max_row: {max_row_reduction:.1f}%, "
    #       f"imbalance: {imbalance_reduction:.1f}%")

    print(f"Crossbar totals from rows - reduction in max_xbar(max write count in a row):  {max_xbar_from_rows_reduction:.1f}%")
    print(f"Crossbar totals from rows - reduction in min_xbar:                            {min_xbar_from_rows_reduction:.1f}%")
    print(f"Crossbar totals from rows - reduction in avg_xbar:                            {avg_xbar_from_rows_reduction:.1f}%")
    print(f"Total row swaps (with TIWL):                                                  {stats_yes.get('total_row_swaps', 0)}")

In [ ]:
def plot_row_wear_heatmap_fixed_bg(pim_simulator_no_tiwl, pim_simulator_with_tiwl,
                          grid_shape=(16, 32), rows_to_show=128, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    import copy

    def build_grid(sim):
        wear_rows = sim.row_write_counts
        # Standard int array with 0 padding (Original Logic)
        C, R = wear_rows.shape
        gr, gc = grid_shape
        total = gr * gc
        take = min(C, total)

        block = wear_rows[:take, :min(rows_to_show, R)]

        if block.shape[1] < rows_to_show:
            pad = np.zeros((block.shape[0], rows_to_show - block.shape[1]), dtype=block.dtype)
            block = np.concatenate([block, pad], axis=1)

        blocks = []
        idx = 0
        for _ in range(gr):
            row_blocks = []
            for _ in range(gc):
                if idx < take:
                    row_blocks.append(block[idx][:, None])
                else:
                    row_blocks.append(np.zeros((rows_to_show, 1), dtype=block.dtype))
                idx += 1
            row_strip = np.concatenate(row_blocks, axis=1)
            blocks.append(row_strip)
        return np.concatenate(blocks, axis=0)

    # 1. Build grids (integers with 0s)
    raw_grid_no = build_grid(pim_simulator_no_tiwl)
    raw_grid_yes = build_grid(pim_simulator_with_tiwl)

    # --- MASKING LOGIC ---
    def mask_zeros(grid):
        # Convert to float to hold NaNs
        grid_float = grid.astype(float)
        # Replace ALL 0s (both data-0 and padding-0) with NaN
        grid_float[grid_float == 0] = np.nan
        # Create a masked array where NaN is invalid
        return np.ma.masked_invalid(grid_float)

    grid_no = mask_zeros(raw_grid_no)
    grid_yes = mask_zeros(raw_grid_yes)

    # --- STYLE & COLORMAP ---
    params = {
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'text.color': 'black',
        'axes.labelcolor': 'black',
        'xtick.color': 'black',
        'ytick.color': 'black',
    }
    plt.rcParams.update(params)

    my_cmap = copy.copy(plt.cm.hot)
    # Set the 'bad' (masked) values to Light Gray.
    # This will cover both the empty padding AND rows with 0 writes.
    my_cmap.set_bad(color='white')

    # --- PLOTTING ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    # Force opaque background for Colab
    fig.patch.set_alpha(1.0)

    # Calculate shared limits based on NON-NAN values only
    # We use nanmin/nanmax so the NaNs don't break the scale
    # If a grid is all NaNs (no writes), default to 0-1 range to avoid errors
    min_no = np.nanmin(grid_no) if not np.all(np.isnan(grid_no)) else 0
    max_no = np.nanmax(grid_no) if not np.all(np.isnan(grid_no)) else 1
    min_yes = np.nanmin(grid_yes) if not np.all(np.isnan(grid_yes)) else 0
    max_yes = np.nanmax(grid_yes) if not np.all(np.isnan(grid_yes)) else 1

    global_min = min(min_no, min_yes)
    global_max = max(max_no, max_yes)

    im1 = ax1.imshow(grid_no, cmap=my_cmap, vmin=global_min, vmax=global_max,
                     interpolation='nearest', aspect='auto')
    ax1.set_title('Row Wear Without TIWL', color='black')
    ax1.set_xlabel('Crossbar column index', color='black')
    ax1.set_ylabel('Row index within each crossbar block', color='black')

    # Colorbar
    cbar1 = plt.colorbar(im1, ax=ax1)
    cbar1.set_label('Writes per row', rotation=270, labelpad=20, color='black')
    cbar1.ax.yaxis.set_tick_params(color='black')
    plt.setp(plt.getp(cbar1.ax.axes, 'yticklabels'), color='black')

    im2 = ax2.imshow(grid_yes, cmap=my_cmap, vmin=global_min, vmax=global_max,
                     interpolation='nearest', aspect='auto')
    ax2.set_title('Row Wear With TIWL', color='black')
    ax2.set_xlabel('Crossbar column index', color='black')
    ax2.set_ylabel('Row index within each crossbar block', color='black')

    # Colorbar
    cbar2 = plt.colorbar(im2, ax=ax2)
    cbar2.set_label('Writes per row', rotation=270, labelpad=20, color='black')
    cbar2.ax.yaxis.set_tick_params(color='black')
    plt.setp(plt.getp(cbar2.ax.axes, 'yticklabels'), color='black')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    # --- STATISTICS (Use original raw integers) ---
    stats_no = pim_simulator_no_tiwl.get_row_wear_statistics()
    stats_yes = pim_simulator_with_tiwl.get_row_wear_statistics()

    def pct_reduction(a, b):
        return 0.0 if a <= 0 else 100.0 * (a - b) / a

    max_xbar_from_rows_reduction = pct_reduction(
        stats_no.get('max_xbar_wear_from_rows', 0), stats_yes.get('max_xbar_wear_from_rows', 0)
    )
    min_xbar_from_rows_reduction = pct_reduction(
        stats_no.get('min_xbar_wear_from_rows', 0), stats_yes.get('min_xbar_wear_from_rows', 0)
    )
    avg_xbar_from_rows_reduction = pct_reduction(
        stats_no.get('avg_xbar_wear_from_rows', 0), stats_yes.get('avg_xbar_wear_from_rows', 0)
    )

    print(f"Crossbar totals from rows - reduction in max_xbar: {max_xbar_from_rows_reduction:.1f}%")
    print(f"Crossbar totals from rows - reduction in min_xbar: {min_xbar_from_rows_reduction:.1f}%")
    print(f"Crossbar totals from rows - reduction in avg_xbar: {avg_xbar_from_rows_reduction:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_crossbar_wear_heatmap(pim_simulator_no_tiwl, pim_simulator_with_tiwl,
                              grid_shape=(32, 32), save_path=None):
    """
    Plot side-by-side heatmaps comparing crossbar wear with/without TIWL

    Args:
        pim_simulator_no_tiwl: PIMSimulator instance without TIWL
        pim_simulator_with_tiwl: PIMSimulator instance with TIWL
        grid_shape: Tuple (rows, cols) for arranging crossbars in grid
        save_path: Optional path to save the plot
    """


    # Reshape to 2D grid for visualization
    grid_rows, grid_cols = grid_shape
    # grid_rows * grid_cols = 32 * 32 = 1024(xbars total)


    # Create side-by-side subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    # Heatmap without wear levelling
    if pim_simulator_no_tiwl:
      wear_without_tiwl = pim_simulator_no_tiwl.twc_counters
      wear_grid_no_tiwl = wear_without_tiwl[:grid_rows*grid_cols].reshape(grid_rows, grid_cols)

      im1 = ax1.imshow(wear_grid_no_tiwl, cmap='hot', interpolation='nearest')
      ax1.set_title('Last Interval Wear Distribution Without TIWL', fontsize=14, fontweight='bold')
      ax1.set_xlabel('Crossbar Column Index')
      ax1.set_ylabel('Crossbar Row Index')
      cbar1 = plt.colorbar(im1, ax=ax1)
      cbar1.set_label('Max Write Count per Crossbar', rotation=270, labelpad=20)

    # Heatmap with wear levelling enabled
    if pim_simulator_with_tiwl:
      # Extract wear data (twc_counters contains max writes per crossbar)
      wear_with_tiwl = pim_simulator_with_tiwl.twc_counters
      wear_grid_with_tiwl = wear_with_tiwl[:grid_rows*grid_cols].reshape(grid_rows, grid_cols)

      im2 = ax2.imshow(wear_grid_with_tiwl, cmap='hot', interpolation='nearest')
      ax2.set_title('Last Interval Wear Distribution With TIWL', fontsize=14, fontweight='bold')
      ax2.set_xlabel('Crossbar Column Index')
      ax2.set_ylabel('Crossbar Row Index')
      cbar2 = plt.colorbar(im2, ax=ax2)
      cbar2.set_label('Max Write Count per Crossbar', rotation=270, labelpad=20)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

    # Print statistics
    if pim_simulator_no_tiwl:
      print(f"Without TIWL - Max: {wear_without_tiwl.max():,}, Min: {wear_without_tiwl.min():,}")
    if pim_simulator_with_tiwl:
      print(f"With TIWL    - Max: {wear_with_tiwl.max():,}, Min: {wear_with_tiwl.min():,}")
    if pim_simulator_no_tiwl and pim_simulator_with_tiwl:
      print(f"Wear reduction: {((wear_without_tiwl.max() - wear_with_tiwl.max()) / wear_without_tiwl.max() * 100):.1f}%")


In [ ]:
# Load MNIST digits dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
# training config
EPOCHS = 10

from torchsummary import summary

In [ ]:
# Initialize PIM simulator and hooks
pim_sim = PIMSimulator(num_crossbars=512, crossbar_size=(128, 128))
pim_hook = PIMTrainingHook(pim_sim, k_swaps=25)

model = torch.nn.Sequential(
    nn.Flatten(),
    torch.nn.Linear(784, 256),
    torch.nn.ReLU(),
    torch.nn.Linear(256, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10)
)

# Print model summary with input size
# print(summary(model, input_size=(1, 784)), model)

# pim_sim.register_model(model) <----- this is already done in the pimhook __init__

# Register PIM monitoring hooks
# register_hooks internally calls .register_model
pim_hook.register_hooks(model)

# Training loop with PIM simulation
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# loss function
criterion = torch.nn.CrossEntropyLoss()

# print("Parameters", pim_sim.tensor_registry.keys())

pim_sim.logger = pim_sim.log_memory_access
for epoch in range(0, 2):
  for batch_idx, (data, target) in enumerate(train_loader):
          optimizer.zero_grad()


          # Forward pass (monitored by hooks)
          output = model(data)
          loss = criterion(output, target)

          # Backward pass (monitored by hooks)
          loss.backward()

          # Parameter updates (monitored by hooks)
          optimizer.step()
          # for layer in pim_sim.tensor_registry.keys():
          #   plt.hist(pim_sim['layer'], bins=50);
          # plt.show()

          if batch_idx % 100 == 0:
              print(f"Epoch: {epoch}, Batch: {batch_idx}")
              # print(pim_sim.param_update_counter)
              # print(f"Max crossbar wear: {pim_sim.twc_counters.max()}")
              # print(f"Wear distribution std: {pim_sim.twc_counters.std():.2f}")

              # row level(intra xbar stats)
              # row_stats = pim_sim.get_row_wear_statistics()
              # print(f"Max row wear: {row_stats['max_row_wear']}")
              # print(f"Row wear imbalance: {row_stats['row_wear_imbalance']:.2f}x"

pim_sim.redistribute_tensors_by_magnitude(model)
pim_sim.logger = pim_sim.log_memory_access_v2
for epoch in range(2, EPOCHS):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()


        # Forward pass (monitored by hooks)
        output = model(data)
        loss = criterion(output, target)

        # Backward pass (monitored by hooks)
        loss.backward()

        # Parameter updates (monitored by hooks)
        optimizer.step()
        # for layer in pim_sim.tensor_registry.keys():
        #   plt.hist(pim_sim['layer'], bins=50);
        # plt.show()

        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch}, Batch: {batch_idx}")
            # print(pim_sim.param_update_counter)
            # print(f"Max crossbar wear: {pim_sim.twc_counters.max()}")
            # print(f"Wear distribution std: {pim_sim.twc_counters.std():.2f}")

            # row level(intra xbar stats)
            # row_stats = pim_sim.get_row_wear_statistics()
            # print(f"Max row wear: {row_stats['max_row_wear']}")
            # print(f"Row wear imbalance: {row_stats['row_wear_imbalance']:.2f}x")
# Cleanup
pim_hook.cleanup()

# Analysis
print("\nFinal TIWL Statistics:")
print(f"Total memory accesses: {pim_sim.access_count}")
print(f"Average crossbar wear: {pim_sim.twc_counters.mean():.2f}")
print(f"Wear imbalance (max/min): {pim_sim.twc_counters.max() / (pim_sim.twc_counters.min() + 1):.2f}")

Crossbar config: (128, 128), 2048 weights per crossbar
Epoch: 0, Batch: 0
Epoch: 0, Batch: 100
Epoch: 0, Batch: 200
Epoch: 0, Batch: 300
Epoch: 0, Batch: 400
Epoch: 0, Batch: 500
Epoch: 0, Batch: 600
Epoch: 0, Batch: 700
Epoch: 0, Batch: 800
Epoch: 0, Batch: 900
Epoch: 1, Batch: 0
Epoch: 1, Batch: 100
Epoch: 1, Batch: 200
Epoch: 1, Batch: 300
Epoch: 1, Batch: 400
Epoch: 1, Batch: 500
Epoch: 1, Batch: 600
Epoch: 1, Batch: 700
Epoch: 1, Batch: 800
Epoch: 1, Batch: 900
Redistributing model: Top 10% to SLC (1-bit), rest to MLC (4-bit)
Mapping complete. SLC Xbars used: 16/51
Epoch: 2, Batch: 0
Epoch: 2, Batch: 100
Epoch: 2, Batch: 200
Epoch: 2, Batch: 300
Epoch: 2, Batch: 400
Epoch: 2, Batch: 500
Epoch: 2, Batch: 600
Epoch: 2, Batch: 700
Epoch: 2, Batch: 800
Epoch: 2, Batch: 900
Epoch: 3, Batch: 0
Epoch: 3, Batch: 100
Epoch: 3, Batch: 200
Epoch: 3, Batch: 300
Epoch: 3, Batch: 400
Epoch: 3, Batch: 500
Epoch: 3, Batch: 600
Epoch: 3, Batch: 700
Epoch: 3, Batch: 800
Epoch: 3, Batch: 900
Epoch: 

In [ ]:
# Initialize PIM simulator and hooks
pim_simulator_no_wear_levelling = PIMSimulator(num_crossbars=512, crossbar_size=(128, 128), wear_levelling_enabled=False, intra_xbar_wear_levelling_enabled=False)
pim_hook = PIMTrainingHook(pim_simulator_no_wear_levelling)

# Your neural network
model = torch.nn.Sequential(
    nn.Flatten(), # Add this line to flatten the input
    torch.nn.Linear(784, 256),
    torch.nn.ReLU(),
    torch.nn.Linear(256, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10)
)

pim_simulator_no_wear_levelling.register_model(model)
# Register PIM monitoring hooks
pim_hook.register_hooks(model)

# Training loop with PIM simulation
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# loss function
criterion = torch.nn.CrossEntropyLoss()

# set the logger
pim_simulator_no_wear_levelling.logger = pim_simulator_no_wear_levelling.log_memory_access
for epoch in range(EPOCHS):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()

        # Forward pass (monitored by hooks)
        output = model(data)
        loss = criterion(output, target)

        # Backward pass (monitored by hooks)
        loss.backward()

        # Parameter updates (monitored by hooks)
        optimizer.step()

        if batch_idx % 100 == 0:
            # print(f"Epoch: {epoch}, Batch: {batch_idx}")
            # print(pim_sim.param_update_counter)
            # print(f"Max crossbar wear: {pim_simulator_no_wear_levelling.twc_counters.max()}")
            # print(f"Wear distribution std: {pim_simulator_no_wear_levelling.twc_counters.std():.2f}")
            # row_stats = pim_simulator_no_wear_levelling.get_row_wear_statistics()
            # print(f"Max row wear: {row_stats['max_row_wear']}")
            # print(f"Row wear imbalance: {row_stats['row_wear_imbalance']:.2f}x")

# Cleanup
pim_hook.cleanup()

# Analysis
print("\nFinal TIWL Statistics:")
print(f"Total memory accesses: {pim_simulator_no_wear_levelling.access_count}")
print(f"Average crossbar wear: {pim_simulator_no_wear_levelling.twc_counters.mean():.2f}")
print(f"Wear imbalance (max/min): {pim_simulator_no_wear_levelling.twc_counters.max() / (pim_simulator_no_wear_levelling.twc_counters.min() + 1):.2f}")

Crossbar config: (128, 128), 2048 weights per crossbar
Wear levelling disabled
Epoch: 0, Batch: 0
Max crossbar wear: 0
Wear distribution std: 0.00
Epoch: 0, Batch: 100
Max crossbar wear: 0
Wear distribution std: 0.00
Epoch: 0, Batch: 200
Max crossbar wear: 5455872
Wear distribution std: 2271250.40
Epoch: 0, Batch: 300
Max crossbar wear: 5455872
Wear distribution std: 2271250.40
Epoch: 0, Batch: 400
Max crossbar wear: 10928128
Wear distribution std: 4549263.43
Epoch: 0, Batch: 500
Max crossbar wear: 16384000
Wear distribution std: 6820513.76
Epoch: 0, Batch: 600
Max crossbar wear: 16384000
Wear distribution std: 6820513.76
Epoch: 0, Batch: 700
Max crossbar wear: 21839872
Wear distribution std: 9091764.13
Epoch: 0, Batch: 800
Max crossbar wear: 21839872
Wear distribution std: 9091764.13
Epoch: 0, Batch: 900
Max crossbar wear: 27312128
Wear distribution std: 11369777.18
Epoch: 1, Batch: 0
Max crossbar wear: 27312128
Wear distribution std: 11369777.18
Epoch: 1, Batch: 100
Max crossbar wear

In [ ]:
plot_row_wear_heatmap(pim_simulator_no_wear_levelling, pim_sim, grid_shape=(32, 32), save_path="row_heatmap.png")

NameError: name 'plot_row_wear_heatmap' is not defined

PIM SIM on CIFAR10 pretrained vgg11 taken from neurosim-dnn repo

In [ ]:
!pip install torchinfo
import torchinfo, torchvision
from torchvision import models
from torchinfo import summary

# Load model
vgg19 = models.vgg19_bn()

# Print summary (requires input size)
summary(vgg19, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [1, 1000]                 --
├─Sequential: 1-1                        [1, 512, 7, 7]            --
│    └─Conv2d: 2-1                       [1, 64, 224, 224]         1,792
│    └─BatchNorm2d: 2-2                  [1, 64, 224, 224]         128
│    └─ReLU: 2-3                         [1, 64, 224, 224]         --
│    └─Conv2d: 2-4                       [1, 64, 224, 224]         36,928
│    └─BatchNorm2d: 2-5                  [1, 64, 224, 224]         128
│    └─ReLU: 2-6                         [1, 64, 224, 224]         --
│    └─MaxPool2d: 2-7                    [1, 64, 112, 112]         --
│    └─Conv2d: 2-8                       [1, 128, 112, 112]        73,856
│    └─BatchNorm2d: 2-9                  [1, 128, 112, 112]        256
│    └─ReLU: 2-10                        [1, 128, 112, 112]        --
│    └─Conv2d: 2-11                      [1, 128, 112, 112]        147,

In [ ]:
# import torch
import torchvision.models as models
# from torchvision import datasets, transforms

# 1. Load pretrained model for CIFAR-10
model = models.vgg19_bn(pretrained=False)
model.classifier[6] = torch.nn.Linear(4096, 10)  # Adjust for CIFAR-10
# Load pretrained weights (download from NeuroSim or train quickly)
# checkpoint = torch.load('./VGG8.pth')
# model.load_state_dict(checkpoint)

# 2. Setup CIFAR-10 data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = datasets.CIFAR10(root='./data', train=True,
                                  download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=64, shuffle=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
100%|██████████| 170M/170M [00:04<00:00, 39.5MB/s]


In [ ]:

# Initialize one simulator with default settings(wear levelling enabled)
pim_sim_cifar = PIMSimulator(num_crossbars=1000000)
pim_sim_cifar.register_model(model)
pim_hook = PIMTrainingHook(pim_sim_cifar, k_swaps=25)
pim_hook.register_hooks(model)

# Run SHORT fine-tuning to capture write pattern
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

NUM_EPOCHS_TO_PROFILE = 2  # Just 2 epochs to capture pattern
for epoch in range(NUM_EPOCHS_TO_PROFILE):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()  # This triggers your hooks
        optimizer.step()

        if batch_idx >= 100:  # Sample first 100 batches
            print(f"Epoch: {epoch}, Batch: {batch_idx}")
            print(pim_sim_cifar.param_update_counter)
            break

# 5. Analyze pattern captured
print(f"Total accesses in 2 epochs: {pim_sim_cifar.access_count}")
print(f"Max crossbar wear: {pim_sim_cifar.twc_counters.max()}")
print(f"Wear imbalance: {pim_sim_cifar.twc_counters.max() / (pim_sim_cifar.twc_counters.min() + 1):.2f}")

# 6. Optionally: Loop the captured pattern to simulate longer training
# The write distribution won't change much after initial epochs

Crossbar config: (128, 128), 2048 weights per crossbar


ValueError: logger not set

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

def prepare_partitioned_model(model, k_percent=10):
    partitioned_weights = {}
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            w = module.weight.data
            abs_w = torch.abs(w)
            flattened = abs_w.view(-1)
            num_elements = flattened.size(0)

            # Sampling for large tensors to avoid memory errors
            if num_elements > 1_000_000:
                indices = torch.randint(0, num_elements, (1_000_000,))
                sample_for_threshold = flattened[indices]
            else:
                sample_for_threshold = flattened

            threshold = torch.quantile(sample_for_threshold, 1.0 - (k_percent / 100.0))

            mask = abs_w >= threshold
            w_slc = torch.where(mask, w, torch.tensor(0.0).to(w.device))
            w_mlc = torch.where(~mask, w, torch.tensor(0.0).to(w.device))

            # --- Statistics & Visualization Section ---
            slc_mags = abs_w[mask].cpu().numpy()
            mlc_mags = abs_w[~mask].cpu().numpy()

            print(f"\nLayer: {name}")
            print(f"  Threshold Magnitude: {threshold:.6f}")
            print(f"  SLC (Top {k_percent}%): Mean Mag = {slc_mags.mean():.6f}, Max = {slc_mags.max():.6f}")
            print(f"  MLC (Rest): Mean Mag = {mlc_mags.mean():.6f}, Min = {mlc_mags.min():.6f}")

            # Plotting the distribution for the first few layers
            # if len(partitioned_weights) < 3:
            plt.figure(figsize=(10, 4))
            plt.hist(mlc_mags, bins=50, alpha=0.5, label='MLC (High Density)', color='blue')
            plt.hist(slc_mags, bins=50, alpha=0.7, label='SLC (High Precision)', color='orange')
            plt.axvline(threshold.cpu(), color='red', linestyle='dashed', linewidth=2, label='Threshold')
            plt.title(f"Weight Magnitude Distribution: {name}")
            plt.xlabel("Absolute Magnitude")
            plt.ylabel("Frequency")
            plt.legend()
            plt.show()
            # ------------------------------------------

            partitioned_weights[name] = {
                'slc': w_slc,
                'mlc': w_mlc,
                'bias': module.bias.data if module.bias is not None else None
            }
    return partitioned_weights

In [ ]:
partitioned_weights = prepare_partitioned_model(model, 10)

In [ ]:
# import torch
# import torchvision.models as models
# from torchvision import datasets, transforms

# 1. Load pretrained model for CIFAR-10
# model = models.vgg11_bn(pretrained=False)
# model.classifier[6] = torch.nn.Linear(4096, 10)  # Adjust for CIFAR-10
# # Load pretrained weights (download from NeuroSim or train quickly)
# checkpoint = torch.load('./VGG8.pth')
# model.load_state_dict(checkpoint)

# 2. Setup CIFAR-10 data
# transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
# ])
# train_dataset = datasets.CIFAR10(root='./data', train=True,
#                                   download=True, transform=transform)
# train_loader = torch.utils.data.DataLoader(train_dataset,
                                          #  batch_size=64, shuffle=True)
# already imported above

# Initialize one simulator with default settings(wear levelling enabled)
pim_sim_cifar_no_wear_levelling = PIMSimulator(num_crossbars=1000000, wear_levelling_enabled=False)
pim_sim_cifar_no_wear_levelling.register_model(model)
pim_hook = PIMTrainingHook(pim_sim_cifar_no_wear_levelling, k_swaps=25)
pim_hook.register_hooks(model)

# Run SHORT fine-tuning to capture write pattern
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

NUM_EPOCHS_TO_PROFILE = 2  # Just 2 epochs to capture pattern
for epoch in range(NUM_EPOCHS_TO_PROFILE):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()  # This triggers your hooks
        optimizer.step()

        if batch_idx >= 100:  # Sample first 100 batches
            break

# 5. Analyze pattern captured
print(f"Total accesses in 2 epochs: {pim_sim_cifar_no_wear_levelling.access_count}")
print(f"Max crossbar wear: {pim_sim_cifar_no_wear_levelling.twc_counters.max()}")
print(f"Wear imbalance: {pim_sim_cifar_no_wear_levelling.twc_counters.max() / (pim_sim_cifar_no_wear_levelling.twc_counters.min() + 1):.2f}")

* The data for row_write_counts[0] (the row wear for physical rows of crossbar 0) will form the vertical strip at x=0, from y=0 to y=127.

* The color at (x, y) represents the value of `row_write_counts[(y // 128) * 32 + x][y % 128]`
* The crossbar index is: "(y // rows_to_show) * grid_cols + x". The physical row index within that crossbar is "y % rows_to_show".

In [ ]:
plot_row_wear_heatmap(pim_sim_cifar_no_wear_levelling, pim_sim_cifar, grid_shape=(1000, 1000), save_path="row_heatmap.png")

In [ ]:
plot_crossbar_wear_heatmap(pim_sim_cifar_no_wear_levelling, pim_sim_cifar, grid_shape=(1000, 1000), save_path="heatmap.png")

In [ ]:
# test accuracy of model on cifar10
